# Conjugation boxes

**Download Notebook** - {nb-download}`conjugation_box.ipynb`

Build an XYZ Pauli gadget and a two-term LCU, then apply call, dagger, control,
and controlled dagger through `ConjugationBox`. Nest boxes using a unitary
function wrapper.

Run cells in order. Install dependencies with `uv sync --extra dev-dependencies`.

In [1]:
import numpy as np
import zixy.qubit.pauli as zqp
from scipy.linalg import expm

from guppylang import guppy
from guppylang.std.angles import angle
from guppylang.std.builtins import array, control, dagger
from guppylang.std.quantum import cx, cz, h, qubit, ry, rz, sdg, x

from guppyalgos.primitives.subroutines.conjugation import ConjugationBox
from guppyalgos.testing import assert_unitary_modifiers

# Guppy uses half-turns; alpha is the angle in radians.
theta = 0.37
alpha = np.pi * theta

## What does ConjugationBox do?

G must be daggerable and U unitary. Both act on the same register.
`box.compose(qs)` executes **G → U → G†**; matrices act right to left:

$$
W=G^{\dagger}UG,
\qquad
W^{\dagger}=G^{\dagger}U^{\dagger}G.
$$

| Call context | Execution order |
| --- | --- |
| Ordinary call | G → U → G† |
| `with dagger:` | G → U† → G† |
| `with control(controls):` | G → controlled-U → G† |
| `with control(controls), dagger:` | G → controlled-U† → G† |

Only U receives the external control c:

$$
\mathrm{C}(W)=(I_c\otimes G^{\dagger})\,\mathrm{C}(U)\,(I_c\otimes G).
$$

- c = 0: U is skipped and G† G cancels.
- c = 1: the box applies W.

Use U† in place of U for the controlled adjoint.

### How the checks work

The explicit functions show how to use each modifier. For validation,
`assert_unitary_modifiers` simulates the ordinary call, dagger, control, and
controlled dagger against the reference matrix, including relative control phases.

## 2. An XYZ Pauli gadget

For $P=X_0Y_1Z_2$, G applies H on q0, S† then H on q1, and two parity CNOTs.
This gives $G^{\dagger}Z_2G=P$. With $U=R_z(\alpha)$ on q2 and
$\alpha=\pi\theta$:

$$
W_{XYZ}(\theta)=G^{\dagger}R_z(\alpha)G=e^{-i\alpha X_0Y_1Z_2/2}.
$$

In [2]:
@guppy(daggerable=True)
def g(qs: array[qubit, 3]) -> None:
    h(qs[0])
    sdg(qs[1])
    h(qs[1])
    cx(qs[0], qs[2])
    cx(qs[1], qs[2])


@guppy(unitary=True)
def u(qs: array[qubit, 3]) -> None:
    rz(qs[2], angle(theta))

### One function for each modifier

Only the middle $\mathrm{C}R_z(\alpha)$ receives the external control;
the basis changes and parity CNOTs remain uncontrolled.

In [3]:
@guppy(unitary=True)
def pauli_gadget(qs: array[qubit, 3]) -> None:
    box = ConjugationBox(g, u)
    box.compose(qs)


@guppy
def pauli_daggered(qs: array[qubit, 3]) -> None:
    box = ConjugationBox(g, u)
    with dagger:
        box.compose(qs)


@guppy
def pauli_controlled(controls: array[qubit, 1], qs: array[qubit, 3]) -> None:
    box = ConjugationBox(g, u)
    with control(controls):
        box.compose(qs)


@guppy
def pauli_ctrl_daggered(controls: array[qubit, 1], qs: array[qubit, 3]) -> None:
    box = ConjugationBox(g, u)
    with control(controls), dagger:
        box.compose(qs)

### Check all modifiers

Build $W=e^{-i\alpha X_0Y_1Z_2/2}$ with Zixy in big-endian order, then check
all four modes with `assert_unitary_modifiers`.

In [4]:
pauli_hamiltonian = zqp.RealTermSum.from_str("(1.0, X0 Y1 Z2)", 3)
pauli_matrix = pauli_hamiltonian.to_sparse_matrix(big_endian=True).toarray()
pauli_reference = expm(-0.5j * alpha * pauli_matrix)

assert_unitary_modifiers(pauli_gadget, 3, expected_unitary=pauli_reference)
print("All Pauli gadget modifiers passed.")

All Pauli gadget modifiers passed.


## 3. Nested conjugation boxes

Wrap an inner box's `compose` call in a `@guppy(unitary=True)` function,
then use that function as the outer box's U. Both boxes share the same register.

Here $W_1=H R_z(\alpha)H=R_x(\alpha)$ and $G_1=R_y(\alpha/2)$, so

$$
W_2=G_1^{\dagger}W_1G_1
=\exp\!\left[-\frac{i\alpha}{2}
\left(\cos(\alpha/2)X+\sin(\alpha/2)Z\right)\right].
$$

- External control passes through both boxes to the central Rz only.
- The outer Ry and inner Hadamard basis changes run unconditionally.
- Dagger replaces Rz by its adjoint.
- Repeat the wrapper pattern for further levels: $W_{k+1}=G_k^{\dagger}W_kG_k$.


In [5]:
@guppy(daggerable=True)
def inner_basis(qs: array[qubit, 1]) -> None:
    h(qs[0])


@guppy(unitary=True)
def inner_rotation(qs: array[qubit, 1]) -> None:
    rz(qs[0], angle(theta))


@guppy(unitary=True)
def inner_conjugation(qs: array[qubit, 1]) -> None:
    inner = ConjugationBox(inner_basis, inner_rotation)
    inner.compose(qs)


@guppy(daggerable=True)
def outer_basis(qs: array[qubit, 1]) -> None:
    ry(qs[0], angle(theta / 2))


@guppy(unitary=True)
def nested_conjugation(qs: array[qubit, 1]) -> None:
    outer = ConjugationBox(outer_basis, inner_conjugation)
    outer.compose(qs)


In [6]:
nested_hamiltonian = zqp.RealTermSum.from_str(
    f"({np.cos(alpha / 2)}, X0), ({np.sin(alpha / 2)}, Z0)", 1
)
nested_matrix = nested_hamiltonian.to_sparse_matrix(big_endian=True).toarray()
nested_reference = expm(-0.5j * alpha * nested_matrix)
assert_unitary_modifiers(nested_conjugation, 1, expected_unitary=nested_reference)
print("All nested conjugation modifiers passed.")

All nested conjugation modifiers passed.


## 4. A two-term LCU block encoding

Use `qs[0]` as index a and `qs[1]` as target t. G = PREPARE rotates a:

$$
R_y(\alpha)|0\rangle=c_0|0\rangle+c_1|1\rangle,
\qquad
c_0=\cos(\alpha/2),\quad c_1=\sin(\alpha/2).
$$

U = SELECT applies X for index zero and Z for index one:

$$
\mathrm{SELECT}=|0\rangle\langle0|_a\otimes X_t
+|1\rangle\langle1|_a\otimes Z_t.
$$

The box implements

$$
B=\mathrm{PREPARE}^{\dagger}\,\mathrm{SELECT}\,\mathrm{PREPARE},
$$

with index-zero block

$$
(\langle0|_a\otimes I_t)\,B\,(|0\rangle_a\otimes I_t)
=c_0^2X_t+c_1^2Z_t=A.
$$

B is unitary; A need not be. Keep the index qubit when checking B's modifiers.

In [7]:
@guppy(daggerable=True)
def prepare(qs: array[qubit, 2]) -> None:
    ry(qs[0], angle(theta))


@guppy(unitary=True)
def select(qs: array[qubit, 2]) -> None:
    # Open-index control: apply X to the target when the index is zero.
    x(qs[0])
    cx(qs[0], qs[1])
    x(qs[0])
    # Closed-index control: apply Z to the target when the index is one.
    cz(qs[0], qs[1])

### Distinguish the index from the external control

Index a chooses X or Z. External control c turns the whole SELECT block on or off.

- c = 0: SELECT is skipped and PREPARE† PREPARE cancels.
- c = 1: SELECT uses a to choose the Pauli.
- PREPARE and PREPARE† always run.

The control covers SELECT's open-index X gates too. It controls the full block
encoding B, including its index qubit, rather than A alone.

In [8]:
@guppy(unitary=True)
def lcu(qs: array[qubit, 2]) -> None:
    box = ConjugationBox(prepare, select)
    box.compose(qs)


@guppy
def lcu_daggered(qs: array[qubit, 2]) -> None:
    box = ConjugationBox(prepare, select)
    with dagger:
        box.compose(qs)


@guppy
def lcu_controlled(controls: array[qubit, 1], qs: array[qubit, 2]) -> None:
    box = ConjugationBox(prepare, select)
    with control(controls):
        box.compose(qs)


@guppy
def lcu_ctrl_daggered(controls: array[qubit, 1], qs: array[qubit, 2]) -> None:
    box = ConjugationBox(prepare, select)
    with control(controls), dagger:
        box.compose(qs)

### Check the four unitary modes

Construct PREPARE and SELECT with Zixy, using

$$
\mathrm{SELECT}=\tfrac12(X_t+Z_aX_t+Z_t-Z_aZ_t),
\qquad B=\mathrm{PREPARE}^{\dagger}\,\mathrm{SELECT}\,\mathrm{PREPARE}.
$$

Check the full block encoding B and its modifiers, retaining the index qubit.

In [9]:
prepare_hamiltonian = zqp.RealTermSum.from_str("(1.0, Y0)", 2)
prepare_matrix = expm(
    -0.5j * alpha * prepare_hamiltonian.to_sparse_matrix(big_endian=True).toarray()
)
select_hamiltonian = zqp.RealTermSum.from_str(
    "(0.5, X1), (0.5, Z0 X1), (0.5, Z1), (-0.5, Z0 Z1)", 2
)
select_matrix = select_hamiltonian.to_sparse_matrix(big_endian=True).toarray()
lcu_reference = prepare_matrix.conj().T @ select_matrix @ prepare_matrix
assert_unitary_modifiers(lcu, 2, expected_unitary=lcu_reference)
print("All LCU modifiers passed.")

All LCU modifiers passed.
